In [1]:
# ─────────────────────────────────────────────────────────────
# COMMIT 1 – Conexión a MongoDB Atlas y lectura de raw_data
# ─────────────────────────────────────────────────────────────
import os, certifi, pandas as pd
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

MONGO_URI            = os.getenv("MONGO_URI")
DB_NAME              = os.getenv("DB_NAME", "Proyecto_Bigdata")
COLLECTION           = os.getenv("RAW_COLLECTION", "Registros_Scraping")
PROCESSED_COLLECTION = os.getenv("PROCESSED_COLLECTION", "processed_data")

if not MONGO_URI:
    raise ValueError("MONGO_URI no encontrado. Verifica el archivo .env")

# JARs disponibles en la carpeta jars/ del proyecto
JARS_DIR = "/home/jovyan/work/jars"
jar_list = [
    "mongo-spark-connector_2.12-10.3.0.jar",
    "bson-4.11.1.jar",
    "mongodb-driver-core-4.11.1.jar",
    "mongodb-driver-sync-4.11.1.jar",
]
jar_paths = ",".join([os.path.join(JARS_DIR, j) for j in jar_list])
os.environ["PYSPARK_SUBMIT_ARGS"] = f"--jars {jar_paths} pyspark-shell"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, trim, lower, length, lit, concat_ws

try:
    spark.stop()
except:
    pass

spark = SparkSession.builder \
    .appName("Processor_ContenedorB") \
    .config("spark.sql.caseSensitive", "true") \
    .getOrCreate()

print("SparkSession lista:", spark.version)

# Leer desde Atlas con PyMongo
print("Leyendo desde raw_data...")
client = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
db     = client[DB_NAME]
datos  = list(db[COLLECTION].find({}, {"_id": 0}))
client.close()
print(f"Registros leidos: {len(datos)}")

# Crear Spark DataFrame desde pandas
df_pandas = pd.DataFrame(datos)
for c in ["titulo_cargo","empresa","pais","descripcion","modalidad","tipo_horario","fecha_captura","fecha_publicacion","grupo"]:
    if c not in df_pandas.columns:
        df_pandas[c] = "No especificado"

df = spark.createDataFrame(df_pandas).select(
    col("titulo_cargo").alias("titulo"),
    col("empresa"), col("pais"), col("fecha_captura"),
    col("descripcion"), col("modalidad"), col("tipo_horario"),
    col("fecha_publicacion"), col("grupo")
)

print(f"Registros listos para procesamiento: {df.count()}")
df.printSchema()


SparkSession lista: 3.5.0
Leyendo desde raw_data...
Registros leidos: 6098
Registros listos para procesamiento: 6098
root
 |-- titulo: string (nullable = true)
 |-- empresa: string (nullable = true)
 |-- pais: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- modalidad: string (nullable = true)
 |-- tipo_horario: string (nullable = true)
 |-- fecha_publicacion: string (nullable = true)
 |-- grupo: string (nullable = true)



In [2]:
# ─────────────────────────────────────────────────────────────
# Integración outliers.ipynb — corregir_outliers()
# ─────────────────────────────────────────────────────────────
from pyspark.sql.functions import length

TITULOS_RUIDO = ["Anuncios Google", "Open job preview"]

def corregir_outliers(df):
    """Elimina ruido de scraping y títulos con largo extremo (IQR)."""
    antes = df.count()

    # 1. Eliminar títulos de ruido conocido
    for ruido in TITULOS_RUIDO:
        df = df.filter(~col("titulo").contains(ruido))

    # 2. Calcular largo del título
    df = df.withColumn("largo_titulo", length(col("titulo")))

    # 3. IQR sobre largo_titulo
    Q1, Q3 = df.approxQuantile("largo_titulo", [0.25, 0.75], 0.01)
    IQR = Q3 - Q1
    limite_superior = Q3 + 3 * IQR

    print(f"Q1={Q1} | Q3={Q3} | IQR={IQR} | Limite superior titulo: {limite_superior}")

    df = df.filter(col("largo_titulo") <= limite_superior)
    df = df.drop("largo_titulo")

    despues = df.count()
    print(f"Outliers eliminados: {antes - despues} ({antes} → {despues})")
    return df

df = corregir_outliers(df)
print(f"Registros tras corregir_outliers: {df.count()}")

Q1=16.0 | Q3=46.0 | IQR=30.0 | Limite superior titulo: 136.0
Outliers eliminados: 60 (6098 → 6038)
Registros tras corregir_outliers: 6038


In [3]:
# ─────────────────────────────────────────────────────────────
# COMMIT 2 – Lógica de limpieza: nulos, duplicados, outliers
# ─────────────────────────────────────────────────────────────
from pyspark.sql.functions import concat_ws

print(f"Registros antes de limpieza: {df.count()}")

# ── 1. Imputar nulos PRIMERO para que la dedup_key sea precisa ─
df = df.fillna({
    "descripcion":       "sin descripcion",
    "modalidad":         "no especificada",
    "tipo_horario":      "no especificado",
    "fecha_publicacion": "desconocida",
    "grupo":             "sin grupo",
})

# ── 2. Eliminar duplicados ────────────────────────────────────
df = df.withColumn(
    "dedup_key",
    concat_ws("||", col("titulo"), col("empresa"), col("modalidad"), col("tipo_horario"))
)
df = df.dropDuplicates(["dedup_key"]).drop("dedup_key")
print(f"Tras dropDuplicates:           {df.count()}")

# ── 3. Eliminar filas con campos críticos nulos ───────────────
df = df.dropna(subset=["titulo", "empresa", "pais"])
print(f"Tras dropna (campos críticos): {df.count()}")

# ── 4. Normalización de texto ─────────────────────────────────
df = df \
    .withColumn("titulo",       lower(trim(col("titulo")))) \
    .withColumn("empresa",      lower(trim(col("empresa")))) \
    .withColumn("pais",         lower(trim(col("pais")))) \
    .withColumn("modalidad",    lower(trim(col("modalidad")))) \
    .withColumn("tipo_horario", lower(trim(col("tipo_horario"))))

# ── 5. Outliers: descartar descripciones muy cortas ───────────
df = df.filter(length(col("descripcion")) > 5)
print(f"Tras filtro outliers:          {df.count()}")

# ── 6. Vista previa ───────────────────────────────────────────
df.show(5, truncate=60)

# ── 7. Escribir en processed_data con PyMongo ────────────────
registros_limpios = df.toPandas().to_dict("records")
client_w = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
col_proc  = client_w[DB_NAME][PROCESSED_COLLECTION]
col_proc.drop()
col_proc.insert_many(registros_limpios)
total_limpios = col_proc.count_documents({})
client_w.close()
print(f"OK {total_limpios} registros guardados en '{PROCESSED_COLLECTION}'")

Registros antes de limpieza: 6038
Tras dropDuplicates:           3719
Tras dropna (campos críticos): 3719
Tras filtro outliers:          3719
+------------------------------------------------------------+------------------+-----+-------------------+------------------------------------------------------------+----------+---------------+-----------------+--------------+
|                                                      titulo|           empresa| pais|      fecha_captura|                                                 descripcion| modalidad|   tipo_horario|fecha_publicacion|         grupo|
+------------------------------------------------------------+------------------+-----+-------------------+------------------------------------------------------------+----------+---------------+-----------------+--------------+
|(212) técnico social para administración y análisis de la...|corporación opción|chile|2026-03-01 03:56:08|(212) Técnico Social para Administración y Análisis de la...|pre

In [4]:
# ─────────────────────────────────────────────────────────────
# COMMIT 3 – Ingeniería de características (columnas derivadas)
# ─────────────────────────────────────────────────────────────
import certifi
from pymongo import MongoClient
from pyspark.sql.functions import col, lower, when, length, lit

# Leer processed_data con PyMongo
print("Leyendo processed_data desde Atlas...")
client_r = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
datos_proc = list(client_r[DB_NAME][PROCESSED_COLLECTION].find({}, {"_id": 0}))
client_r.close()
df = spark.createDataFrame(pd.DataFrame(datos_proc))
print(f"Registros leidos de processed_data: {df.count()}")

# ── Columna 1: es_ti ──────────────────────────────────────────
keywords_ti = [
    "developer", "desarrollador", "programador", "software", "data",
    "cloud", "devops", "sre", "backend", "frontend", "fullstack",
    "ingeniero de software", "analista de sistemas", "qa", "tester",
    "machine learning", "inteligencia artificial", "bi ", "base de datos"
]
condicion_ti = lower(col("titulo")).rlike("|".join(keywords_ti))
df = df.withColumn("es_ti", when(condicion_ti, lit(True)).otherwise(lit(False)))

# ── Columna 2: nivel_seniority ────────────────────────────────
df = df.withColumn(
    "nivel_seniority",
    when(lower(col("titulo")).rlike("senior|sr\\.|lead|principal|jefe|coordinador"), "Senior")
    .when(lower(col("titulo")).rlike("semi|ssr\\.|mid|middle"), "Semi-Senior")
    .when(lower(col("titulo")).rlike("junior|jr\\.|trainee|practicante|pasante|intern"), "Junior")
    .otherwise("No especificado")
)

# ── Columna 3: es_remoto ──────────────────────────────────────
df = df.withColumn(
    "es_remoto",
    when(col("modalidad").isin("remoto", "teletrabajo", "hibrido"), lit(True))
    .otherwise(lit(False))
)

# ── Columna 4: largo_descripcion ─────────────────────────────
df = df.withColumn("largo_descripcion", length(col("descripcion")))

# ── Columna 5: categoria_horario_modalidad ────────────────────
df = df.withColumn(
    "categoria_horario_modalidad",
    when((col("modalidad") == "remoto")      & (col("tipo_horario") == "full time"),  "Remoto FT")
    .when((col("modalidad") == "remoto")     & (col("tipo_horario") == "part time"),  "Remoto PT")
    .when((col("modalidad") == "presencial") & (col("tipo_horario") == "full time"), "Presencial FT")
    .when((col("modalidad") == "presencial") & (col("tipo_horario") == "part time"), "Presencial PT")
    .when(col("modalidad") == "hibrido", "Hibrid")
    .otherwise("Otro")
)

# ── Vista previa ──────────────────────────────────────────────
df.select(
    "titulo", "es_ti", "nivel_seniority", "es_remoto",
    "largo_descripcion", "categoria_horario_modalidad"
).show(10, truncate=40)

print("\nDistribución es_ti:")
df.groupBy("es_ti").count().show()

print("Distribución nivel_seniority:")
df.groupBy("nivel_seniority").count().orderBy("count", ascending=False).show()

print("Distribución categoria_horario_modalidad:")
df.groupBy("categoria_horario_modalidad").count().orderBy("count", ascending=False).show()

# ── Escribir con pymongo (más confiable que el conector Spark) ─
print("\nConvirtiendo a pandas para escritura...")
pandas_df = df.toPandas()

# Convertir booleanos de numpy a Python nativo (evita errores en MongoDB)
pandas_df["es_ti"]     = pandas_df["es_ti"].astype(bool)
pandas_df["es_remoto"] = pandas_df["es_remoto"].astype(bool)

registros = pandas_df.to_dict(orient="records")

client     = MongoClient(MONGO_URI, tlsCAFile=certifi.where())
coleccion  = client[DB_NAME][PROCESSED_COLLECTION]
coleccion.drop()
coleccion.insert_many(registros)
total = coleccion.count_documents({})
client.close()

spark.stop()
print(f"\n✅ processed_data actualizado con columnas derivadas ({total} registros)")

Leyendo processed_data desde Atlas...
Registros leidos de processed_data: 3719
+----------------------------------------+-----+---------------+---------+-----------------+---------------------------+
|                                  titulo|es_ti|nivel_seniority|es_remoto|largo_descripcion|categoria_horario_modalidad|
+----------------------------------------+-----+---------------+---------+-----------------+---------------------------+
|(212) técnico social para administrac...|false|No especificado|    false|              153|              Presencial FT|
|2. ejecutivo/a de ventas individuales...|false|No especificado|    false|              148|              Presencial FT|
| 266930-04 consultor en calidad de aguas|false|No especificado|    false|              255|                       Otro|
|266930-08 consultor ingeniero de gest...|false|No especificado|    false|              266|                       Otro|
|266930-3.1 consultor en calidad y mej...|false|No especificado|    false|